## Week 04: Lab 01 - Abstraction Levels and the building blocks

### Four level of abstractions: 
    1. Langchain-core -- (currently exploring this one)
    2. Langgraph
    3. Langchain (create_agent)
    4. DeepAgent

#### Day 01: The building blocks

In [1]:
# Let's import some libraries
from dotenv import load_dotenv
from langchain_azure_ai.chat_models import AzureAIOpenAIApiChatModel
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from IPython.display import Markdown, display
import gradio as gr
from azure.identity import AzureCliCredential

import json

# let's load the environment variables
load_dotenv(override=True)

c:\Users\hsingh8\OneDrive - London Stock Exchange Group\Documents\Udemy Learning\Agentic Frameworks\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
# Let's see if the API key is working/helping us to call LLM from Azure Foundry
import os
# From OpenAI
AZURE_OPENAI_API_KEY= os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_MODEL_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_GPT_41 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_41")
AZURE_OPENAI_DEPLOYMENT_GPT_54_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_54_mini")
AZURE_OPENAI_DEPLOYMENT_GPT_55 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_55")
AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini")
if AZURE_OPENAI_API_KEY:
    print("AZURE_OPENAI_API_KEY is available")
else:
    print("AZURE_OPENAI_API_KEY is not available")

# From Anthropic
AZURE_CLAUDE_DEPLOYMENT_OPUS_48=os.getenv("AZURE_CLAUDE_DEPLOYMENT_OPUS_48")
AZURE_CLAUDE_ENDPOINT=os.getenv("AZURE_CLAUDE_ENDPOINT")
AZURE_CLAUDE_API_KEY=os.getenv("AZURE_CLAUDE_API_KEY")
if AZURE_CLAUDE_API_KEY:
    print("AZURE_CLAUDE_API_KEY is avaiable")
else:
    print("AZURE_CLAUDE_API_KEY is not available")



AZURE_OPENAI_API_KEY is available
AZURE_CLAUDE_API_KEY is avaiable


In [3]:
# setting up the chat model client
llm  = AzureChatOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    azure_deployment = AZURE_OPENAI_DEPLOYMENT_GPT_41,
    api_key = AZURE_OPENAI_API_KEY,
    api_version = AZURE_OPENAI_API_VERSION
)
llm

AzureChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15', 'langchain-openai': '1.5.1'}}, profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001D35222BC70>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001D305DE9E70>, root_client=<openai.lib.azure.AzureOpenAI object at 0

In [14]:
reply = llm.invoke("Hey, tell me a joke about AI agents.. a good one?")
print(reply.content)

Sure! Here you go:

Why did the AI agent break up with its neural network?

Because it couldn’t handle all the layers of commitment!


In [16]:
## using streaming
for chunk in llm.stream("Tell me good poem about AI agents in 5 lines"):
    print(chunk.content, end = "", flush=True)

Silent algorithms whisper, unseen in the night,  
Agents of logic crafting futures in code-lit light.  
They gather, analyze, and ponder with unseen care,  
Learning from patterns, weaving stories we share.  
AI guides curious minds, bridging worlds lost and rare.

##### Messages

In [17]:
messages = [
    SystemMessage("You are a terse assistant who answers in exactly five words."),
    HumanMessage("What is the capital of France?")
]
print(llm.invoke(messages).content)

messages_in_dicts = [
    {
        'role': 'system',
        'content': 'You are a terse assistant who answers in exactly five words'
    },
    {
        'role': 'user',
        'content': 'What is the capital of Canada?'
    }
]
print(llm.invoke(messages_in_dicts).content)

Paris is France’s capital city.
Ottawa is Canada’s capital city.


##### @tool decorator

In [4]:
@tool
def get_share_price(symbol: str) -> float:
    """Return current share price for a given ticker symbol."""
    fake_prices = {
        "AAPL": 123.4,
        "GOOG": 345.5,
        "AMZN": 198.0,
    }
    return fake_prices.get(symbol.upper(), "Not Available")


In [5]:
print("name:", get_share_price.name)
print("description:", get_share_price.description)
print("Args:", get_share_price.args)
print(get_share_price.invoke({"symbol":"AAPL"}))

name: get_share_price
description: Return current share price for a given ticker symbol.
Args: {'symbol': {'title': 'Symbol', 'type': 'string'}}
123.4


##### Giving tool to the model

In [8]:
llm_with_tools = llm.bind_tools([get_share_price])

response = llm_with_tools.invoke("WHat is the share price of Amazon..?")
print("content:", repr(response.content))
print("tool_calls:", response.tool_calls)

content: ''
tool_calls: [{'name': 'get_share_price', 'args': {'symbol': 'AMZN'}, 'id': 'call_Pa2Ci5OlPVBJPldXXVlBOQxX', 'type': 'tool_call'}]


In [12]:
### Start the conversation and keep the model's tool request in the history
conversation = [
    HumanMessage("What is the share price of Nvidia?")
]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

## Run each requested tool and add its result as a ToolMessage
if ai_message.tool_calls[0]['name'] == "get_share_price":
    result = get_share_price.invoke(ai_message.tool_calls[0]['args'])
    conversation.append(ToolMessage(content=str(result), tool_call_id=ai_message.tool_calls[0]['id']))

final = llm_with_tools.invoke(conversation)
print("Final answer:" , final.content)

Final answer: I'm unable to retrieve the current share price for Nvidia (NVDA) at the moment. If you have a specific source or date in mind, please let me know, or you may want to check a financial news website or trading platform for the most up-to-date information.


##### Structured Output

In [14]:
# Using Pydantic lib to set up the structured output format for LLM
class Company(BaseModel):
    name: str = Field(description="The name of company")
    ticker:str = Field(description="The stock ticker symbol")
    First_CEO:str = Field(description="The name of first CEO of company")
    founded_year: int = Field(description="The year the company was founded")
    
structured_llm = llm.with_structured_output(Company)
company = structured_llm.invoke("Tell me about the Alphabet company..?")
print("Company Info: ", company)
print("\n Company's first CEO: ", company.First_CEO)

Company Info:  name='Alphabet Inc.' ticker='GOOGL' First_CEO='Larry Page' founded_year=2015

 Company's first CEO:  Larry Page
